In [ ]:
import io
from pathlib import Path

import pandas as pd
import requests

url = "https://api.statbank.dk/v1/data"

# Payload matching the NVG1 table view: totals for age, vehicle type,
# and transport type, all trip lengths and all metrics for 2024.
payload = {
    "table": "NVG1",
    "format": "CSV",
    "variables": [
        {"code": "BILALDER", "values": ["IALT"]},  # Vehicle age: total
        {"code": "TYPEVÆGT", "values": ["0000"]},  # Vehicle type/weight: total
        {"code": "KØRART", "values": ["1000"]},  # Transport type: total
        {"code": "Tid", "values": ["2024"]},  # Year
        {"code": "TURKM", "values": ["*"]},  # Trip length: all
        {"code": "ENHED", "values": ["*"]},  # Metric: all
    ],
}

response = requests.post(url, json=payload)
response.raise_for_status()

# Load the long-form API response using the semicolon separator.
raw_df = pd.read_csv(io.StringIO(response.text), sep=";")

# Match the website layout: trip lengths as rows and metrics as columns.
df = raw_df.pivot(index="TURKM", columns="ENHED", values="INDHOLD").reset_index()
df.columns.name = None

trip_length_order = [
    "Ture i alt",
    "Mindre end 15 km",
    "15-29 km",
    "30-49 km",
    "50-74 km",
    "75-99 km",
    "100-124 km",
    "125-149 km",
    "150-199 km",
    "200-249 km",
    "250-299 km",
    "300 km eller mere",
]
metric_order = [
    "TURKM",
    "Ture i alt, 1000",
    "Ture med læs, 1000",
    "Ture uden læs, 1000",
    "Kørte km i alt, 1000 km",
    "Kørte km med læs, 1000 km",
    "Kørte km uden læs, 1000 km",
    "Pålæsset godsmængde (1000 ton)",
    "Transportarbejde (1000 tonkm)",
]

df["TURKM"] = pd.Categorical(df["TURKM"], categories=trip_length_order, ordered=True)
df = df.sort_values("TURKM").reset_index(drop=True)
df = df[metric_order]

# Use stable names and keep the original label separate from numeric filtering data.
column_names = {
    "TURKM": "trip_length",
    "Ture i alt, 1000": "trips_total_1000",
    "Ture med læs, 1000": "trips_loaded_1000",
    "Ture uden læs, 1000": "trips_unloaded_1000",
    "Kørte km i alt, 1000 km": "km_total_1000",
    "Kørte km med læs, 1000 km": "km_loaded_1000",
    "Kørte km uden læs, 1000 km": "km_unloaded_1000",
    "Pålæsset godsmængde (1000 ton)": "freight_loaded_1000_tonnes",
    "Transportarbejde (1000 tonkm)": "transport_work_1000_tonkm",
}
df = df.rename(columns=column_names)
df["trip_length"] = df["trip_length"].astype(str)

# The total row has no single trip length, so leave its lower bound empty.
minimum_length_by_label = {
    "Ture i alt": pd.NA,
    "Mindre end 15 km": 0,
    "15-29 km": 15,
    "30-49 km": 30,
    "50-74 km": 50,
    "75-99 km": 75,
    "100-124 km": 100,
    "125-149 km": 125,
    "150-199 km": 150,
    "200-249 km": 200,
    "250-299 km": 250,
    "300 km eller mere": 300,
}
df["trip_length_min_km"] = df["trip_length"].map(minimum_length_by_label).astype("Int64")
df = df[
    ["trip_length_min_km", "trip_length"]
    + [column for column in column_names.values() if column != "trip_length"]
]

# Example: include the 150-199, 200-249, 250-299, and 300+ km rows.
transport_work_from_150_km = df.loc[
    df["trip_length_min_km"] >= 150,
    "transport_work_1000_tonkm",
].sum()

output_path = Path("data/NVG1_2024.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved {len(df)} rows to {output_path}")
print(f"Transportarbejde for trips >= 150 km: {transport_work_from_150_km:,} (1000 tonkm)")
print(df.to_string(index=False))

In [ ]:
# Calculate transport work for trips shorter than and at least 150 km.
transport_work_150_km_or_more = df.loc[
    df["trip_length_min_km"] >= 150,
    "transport_work_1000_tonkm",
].sum()
transport_work_under_150_km = df.loc[
    df["trip_length_min_km"] < 150,
    "transport_work_1000_tonkm",
].sum()

# Calculate each group's share of the combined transport work.
total_transport_work = (
    transport_work_under_150_km + transport_work_150_km_or_more
)
short_trips_percentage = (
    transport_work_under_150_km / total_transport_work * 100
)
long_trips_percentage = (
    transport_work_150_km_or_more / total_transport_work * 100
)

print(
    "Transport work for trips of 150 km or more: "
    f"{transport_work_150_km_or_more:,} (1000 tonkm)"
)
print(
    "Transport work for trips under 150 km: "
    f"{transport_work_under_150_km:,} (1000 tonkm)"
)
print(
    f"Short trips: {short_trips_percentage:.1f}% | "
    f"Long trips: {long_trips_percentage:.1f}%"
)